# Проект: Анализ отзывов на рестораны и отели

## Источник и состав данных

### Источник:

данные из Яндекс.карт за 2023 год: https://www.kaggle.com/datasets/kyakovlev/yandex-geo-reviews-dataset-2023

### Состав данных:

Адрес организации (address)

Название организации (name_ru)

Категория, которой принадлежит организация (rubrics)

Оценка пользователя от 0 до 5 (rating)

Текст отзыва (text)

## EDA

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', None)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
data = pd.read_csv('drive/MyDrive/Karpov_Courses_ITMO/geo-reviews-dataset-2023.csv')

In [10]:
## Отбираем отели и рестораны
data = data[(data['rubrics'].isin(['Гостиница', 'Ресторан', 'Кафе', 'Быстрое питание', 'Ресторан;Кафе', 'Кафе;Ресторан',
                                 'База, дом отдыха', 'Санаторий', 'Турбаза', 'Ресторан;Бар, паб;Кафе', 'Столовая',
                                 'Бар, паб', 'Ресторан;Бар, паб', 'Быстрое питание;Кафе', 'Кафе;Быстрое питание',
                                 'Ресторан;Банкетный зал', 'Ресторан;Кафе;Банкетный зал', 'Бар, паб;Кафе;Ресторан',
                                 'Ресторан;Доставка еды и обедов', 'Бар, паб;Ресторан', 'Кофейня;Кафе',
                                 'Кафе;Бар, паб;Ресторан', 'Ресторан;Кафе;Доставка еды и обедов',
                                 'Быстрое питание;Кафе;Ресторан', 'Кафе;Бар, паб', 'Кафе;Бар, паб',
                                 'Ресторан;Доставка еды и обедов;Банкетный зал', 'Пиццерия;Быстрое питание',
                                 'Кафе;Ресторан;Доставка еды и обедов', 'Бар, паб;Кафе'])) |
            (data['rubrics'].str.contains('Ресторан')) | (data['rubrics'].str.contains('Кафе')) |
            (data['rubrics'].str.contains('Гостиница'))]

In [11]:
data['org_type'] = ['Гостиница' if ('Гостиница' in x) | (x in ('База, дом отдыха', 'Санаторий', 'Турбаза')) else
                    'Ресторан' for x in data['rubrics']]

In [12]:
data.head()

,address,name_ru,rating,rubrics,text,org_type
6,"Воронежская область, Богучарский район, М-4 Дон, 770-й километр",У тещи,4.0,Кафе,"Глубинка страны во всех своих проявлениях. Ассортимент столовский, интерьер, качество и цены приемлемые для средней бюджетной столовой СССР, не все чистое, все не новое... А что бы вы хотели на трассе. Поел, желудок не бастовал, значит риск был оправдан. Номер для ночлега аналогичного толка. Пластиковое окно нормально не закрылось, шторы на окне нет, только тюль. Для первого этажа - плохо. Мало ли кто заглядывает. Туалет, душ... Совок, но, повторюсь, для трассы да за 1000 руб - сойдет.\n",Ресторан
8,"Москва, 4-й Кожевнический переулок, 4",Jinju,5.0,Кафе;Кофейня,"5 из 5🖤 Пил кофе и в Риме, и в Париже, но вкуснее, чем капуч на фундучном молоке с фирменными сливками Джинжу, не пробовал ничего! Десерты тоже очень необычные. Ребята - бариста большие молодцы! \n\nЧто можно улучшить? Маловато места, с посадкой можно что-то придумать?\n",Ресторан
9,"Москва, 4-й Кожевнический переулок, 4",Jinju,4.0,Кафе;Кофейня,"Не очень удобное расположение, от метро идти мин 20 быстрым шагом через промзону. В самом кофе мест очень мало, а желающих очень много(( пирожные очень вкусные, кофе…бывает вкуснее. Второй раз именно туда на пойду.\n",Ресторан
13,"Краснодарский край, городской округ Сочи, посёлок городского типа Дагомыс, Балтийская улица, 19",Пандок,2.0,Ресторан,"Самый большой плюс это месторасположение, набережная , шикарный вид на море! Красиво, уютно, вот собственно плюсы закончились .. огорчает отношение к посетителям, официанты неприветливые, не здравствуйте вам, не до свидания . Лица недовольные, неприятные, больше не хочется смотреть на такие! Кухня тоже оставляет желать лучшего, в люле кебаб кости попадаются, шашлык из говядины сухой и невкусный. Мы на отдыхе , на позитиве , денег не жалеем, но хочется приходить туда где нам рады!!\n",Ресторан
14,"Краснодарский край, городской округ Сочи, посёлок городского типа Дагомыс, Балтийская улица, 19",Пандок,5.0,Ресторан,"Добрый день! Сегодня во второй раз посетили с дочкой ресторан, нравится меню (куриный суп-лапша, пицца 4 сыра, яблочный штрудель; сациви из курицы, облепиховый чай-то, что успели попробовать за эти 2 посещения) и особенно порадовало прекрасное обслуживание, спасибо Алле за заботу и предупредительность! Разумное соотношение цены и качества, приятная атмосфера в заведении вызывают желание снова посетить и порекомендовать это место. Спасибо, и успехов! С уважением, Оксана и Злата.\n",Ресторан


In [13]:
## Количество отзывов в каждой категории
data.groupby('org_type')['name_ru'].count()

,name_ru
org_type,
Гостиница,51945
Ресторан,101505


In [14]:
data.groupby(['org_type', 'rating'], as_index = False)['name_ru'].count()

,org_type,rating,name_ru
0,Гостиница,0.0,32
1,Гостиница,1.0,1889
2,Гостиница,2.0,1338
3,Гостиница,3.0,3046
4,Гостиница,4.0,7021
5,Гостиница,5.0,38619
6,Ресторан,0.0,38
7,Ресторан,1.0,6806
8,Ресторан,2.0,3422
9,Ресторан,3.0,5644


In [15]:
## Средняя оценка
data.groupby(['org_type'], as_index = False)['rating'].mean()

,org_type,rating
0,Гостиница,4.521898
1,Ресторан,4.422294


In [16]:
## Мода
data.groupby(['org_type'], as_index=False)['rating'].apply(lambda x: x.mode().iloc[0])

,org_type,rating
0,Гостиница,5.0
1,Ресторан,5.0


In [17]:
data[data['rating'] == 0]

,address,name_ru,rating,rubrics,text,org_type
1907,"Республика Карелия, Суоярви, улица Шельшакова, 1",Гостиница Карелия,0.0,Гостиница,"Гостиница расположенна удобно, если отремонтируют номера, будет супер\n",Гостиница
3584,"Краснодарский край, Анапа, улица Пушкина, 30",Кубань,0.0,Санаторий,"Нет растительного масла для салата, нет зубочисток на столах. Номер требует ремонта: шкаф разваливается, дверь не закрывалась вызывали плотника, туалетная комната стеклянная полочка оторвана вызывали плотника починил, телевизор старый пульт разбит и не работает посуду для чая не не выдают (тарелки,ложки, кружки) даже за отдельную плату\n",Гостиница
3615,"Краснодарский край, Туапсинский район, Новомихайловское городское поселение, село Ольгинка, Черноморская улица, 3",Счастливый Хотей,0.0,Гостиница,"Все довольны, и взрослые и дети.\nВкусная еда.\nБанька и бассейны всё замечательно.\nЕздием к вам уже 4 год и привозим к вам ещё новых гостей(делаем вам рекламу) 😁👍номера комфортные, чистые и уютные.\n",Гостиница
8232,"Краснодарский край, Анапа, Виноградная улица, 14Б",Кайт,0.0,Гостиница,"Отличный гостевой дом. Тихо, чисто, удобные номера, есть холодильник, сплит-система, душ и туалет.До моря 3 минуты, по дороге много столовых, рядом магазины Магнит и Пятерочка. Очень хорошие и доброжелательные хозяева.Рекомендую этот дом.\n",Гостиница
15554,"Нижний Новгород, улица Пискунова, 40, корп. 3",Джани ресторани,0.0,Ресторан;Доставка еды и обедов;Кейтеринг,"Вкусно. Много народу, столик стоит бронировать заранее. Завтраки отличные. Хрустящий баклажан, салат Цезарь, хинкали, всё на высоте. Лимонад - супер. Цены тоже, вполне московские. Что-то отметить - отлично. Рутинно - дороговато\n",Ресторан
...,...,...,...,...,...,...
473438,"Ленинградская область, Выборг, проспект Ленина, 9",Солнечный день,0.0,Доставка еды и обедов;Кафе;Столовая,"Готовьтесь к запаху едой на одежде)) Приятно пахнет ядой. Но, уже в начале посещения, наступает разочарование - подносы...жирные, аж из рук уезжают. Это говорит о многом. Экономия салфеток. На столах их нет. Столовые приборы находятся рядом с кассой. То есть все, что есть на верхней одежде, деньгах, руках-всё на них родимых. Столы чистые, без запаха. Есть туалеты (в них не заходила). Отдельно расположена раковина в коридоре, по пути в туалеты, где можно вымыть руки. Мыло есть, раковина чистая. Ну, а теперь о главном, о еде. Мы пробовали рис, тефтели, куриные купаты, компот, круассан, горячий напиток для здоровья(напоминает чай). Хотели взять макароны, но они были такие уставшие, слипшиеся, и уже в перемешку с соусом, что говорит о том, что они не свежие. Рис очень вкусный. Рассыпчатый, не не доваренный, не переваренный. Тефтели очень нежные, вкусные. Но, вот куриные купаты были сухими, с кисловатым вкусом (не стали их есть, только попробовали). Компот вкусный, но ледяной,тк стаканы хранятся в холодильнике. Не спорю, может так и должен храниться, но на улице зима, хоть и погода осенняя, но все равно прохладно. Круассан сухова. Нет той нежности и тянутости, как должно быть. Девушка, которая накладывает еду, тут же берет деньги в руки. Всё это без перчаток. Увы, но с таким количеством ""Но"", я не могу рекомендовать это заведение. И, открылись вроде бы недавно, а уже такой низкий уровень. Очень хочется верить, что услышат и исправятся.\n",Ресторан
475662,"Республика Дагестан, Дербент, улица Батманова, 54",Рояль,0.0,Гостиница,"Отдыхаем сейчас (июль 2023) . Всё нравится! Номер у нас Семейный на четверых, по размерам средний, есть большая кровать, и две кушетки- умещаемся без проблем. В номерах чисто, аккуратно и красиво. Убираются каждый день, каждые 3 дня меняют бельё. Есть два тёплых бассейна, дети вечером резвятся вовсю. Завтраки включены в стоимость, к ним претензий нет никаких. Обед и ужин- легко, но скуден выбор и субъективно дороговато. Парковка на территории, но мы не загоняли машину- снаружи куча камер, и мы в любой момент можем уехать, куда заблагорассудится. Минуса два: дороговато на фоне остальных и море возле отеля не

Видим, что rating = 0 содержит в том числе положителные отзывы. Поэтому их нужно разметить вручную или убрать из обучения.

In [18]:
data[(data['rating'] >= 1) & (data['rating'] <= 2)][['text', 'rating']].head()

,text,rating
13,"Самый большой плюс это месторасположение, набережная , шикарный вид на море! Красиво, уютно, вот собственно плюсы закончились .. огорчает отношение к посетителям, официанты неприветливые, не здравствуйте вам, не до свидания . Лица недовольные, неприятные, больше не хочется смотреть на такие! Кухня тоже оставляет желать лучшего, в люле кебаб кости попадаются, шашлык из говядины сухой и невкусный. Мы на отдыхе , на позитиве , денег не жалеем, но хочется приходить туда где нам рады!!\n",2.0
18,"1. Доставка очень долгая, на рекламной брошюре написано от 500 р доставка бесплатно, а по факту заплатили 100 р за доставку. Чек был на 577р\n2. Долго плевались от нагара который был на сковородке (подноса) с мясом, хруст на зубах не приятен. \nИспорченное мнение от вашего заведения. \n",1.0
76,"Заказали фо-бо и том-ям\nКороче больше не приду к ним. \nВ том яме заявлены черри их нет в том ям идет кокосовое молоко его нет. \nФо-бо воняет чем то не приятно. В фо-бо нет ростков пшеницы, нет кинзы. Спрашивается почему цена как за целое блюдо, но в блюдах не хватает компонентов. Не позортесь. Смените название, на ""собрали из того что было "" Позор руководству производства. \n",1.0
94,"Кроме того, что место красивое и необычное, плюсов нет. Были 1 июня.\nАренда 2500 час, но это просто за иглу. 1000 пробковый сбор, хотя по умолчанию в заведении нет напитков и еды, все приносишь с собой. Сервировка стола платная, при этом сюда входит тарелка,стакан и приборы (Рюмка или бокал за отдельную плату). Посуда самая дешевая, при этом за 1 разбитую единицу штраф-1000. Стол в иглу, которое досталось нам был не закреплен. Т.е крышка и подножья стола существовали отдельно, каждый раз стол болтался, и не разбить там посуду было сложно. Музыкальная колонка тоже попалась «уставшая» громкость не регулировалась. Заказ был на 3 часа+ 1 в подарок, при этом девушка-администратор, насчитала за 4 часа. Прям настаивать пришлось, чтобы сделать перерасчет. Ощущение, что за все пытаются сбить деньги, при этом сервис на 2 балла.\n",2.0
152,"Зашли единожды в это кафе. Заказали с собой еду. Простите ,но так себе. Картошка фри та же самая никакая. В фаст-фуд кортах и то лучше в разы. Паста тоже ( Будто макароны по-флоцки). Вдобавок дома вскрыв коробки обнаружили, что вместо двух разных пицц нам положили две одинаковых. 😖Пиццей в принципе это можно назвать с натяжкой.Будто дешевой ветчины сверху накидали. Вся еда пресная. \nСамое отвратительно место на районе. Не советую к посещению. \nП.с. На негативные отзывы ставят негативные лайки явно не посетители этого «кафе». \n",1.0


Видим, что с низкими оценками у нас негативные отзывы.

Для оценки качества разметки прогоним отзывы через модель, которая выделит отрицательные, положительные отзывы и посмотрим, на сколько они соответствуют оценкам.

In [19]:
print("Загрузка модели...")
classifier = pipeline(
    task="text-classification",
    model="cointegrated/rubert-tiny-sentiment-balanced",
    device=0 if torch.cuda.is_available() else -1
)


Загрузка модели...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny-sentiment-balanced
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/377 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/241k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/468k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [20]:
label_mapping = {
    'LABEL_0': 'neutral',
    'LABEL_1': 'positive',
    'LABEL_2': 'negative'
}


def process_in_batches(texts, batch_size=64):
    results_labels = []
    results_scores = []


    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        predictions = classifier(batch, truncation=True, max_length=512)

        for pred in predictions:
            results_labels.append(label_mapping.get(pred['label'], pred['label']))
            results_scores.append(round(pred['score'], 3))

    return results_labels, results_scores

In [21]:
texts_list = data['text'].tolist()
labels, scores = process_in_batches(texts_list, batch_size=64)

  0%|          | 0/2402 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [22]:
data['sentiment'] = labels
data['sentiment_confidence'] = scores

In [24]:
data.groupby(['rating', 'sentiment'])['name_ru'].count()

rating  sentiment
0.0     negative          3
        neutral           7
        positive         60
1.0     negative       6522
        neutral        1585
        positive        588
2.0     negative       2524
        neutral        1553
        positive        683
3.0     negative       2967
        neutral        3450
        positive       2273
4.0     negative       2174
        neutral        4714
        positive       9801
5.0     negative       6314
        neutral        7906
        positive     100326
Name: name_ru, dtype: int64

In [26]:
data[(data['rating']==1) & (data['sentiment']=='positive')]['text'].head()

,text
1039,"Извините,обычно не пишу отзывы о заведениях,где питаюсь,но здесь не могу промолчать. Молодому человеку,который обслуживал наш стол,большое спасибо,парень действительно молодец,но еда отвратительна,честно,при всем желании найти плюсы и сказать комплименты-не могу. К сожалению,даже не веранде увидели таракана,что не красит абсолютно никаким образом. Надеюсь,что это действительно поможет Вам становится лучше!\n"
1711,"Приветливый и вежливый персонал и вкусная еда. но ресторан теряет 50% клиентов в очереди из-за того, что ожидание составляет 10-20 минут только в ОЧЕРЕДИ, а уже ожидание заказа 10-15 минут, что для людей, которые приходят сюда на свой рабочий обед критично, это не стоит затраченного времени\n"
2212,"Не умеют готовить еду. Надо им куда то сходить, поесть шашлык , впрочем, в любой другой ресторан кавказской кухни. Помидоры, да, вкусные. Хорошо, что не они их не готовят.\n"
3066,"Пришли туда завтракать, так как очень рекомендовал друг. Сказали могу сесть за любой стол, так как все были свободны. Сел за стол, несмотрятна то что стол сервирован, на столу много хлебных крошек. Принесли блюдо, взял вилку - а она грязная (прям засохшая еда от прошлого посетителя была). Другу принесли суп, он там волос выловил ... больше туда никогда не придём!\n"
4949,"Позвонили забронировать столик, покурить кальян компанией, девушка администратор сказала, что столик есть и все отлично)\nМы собрались с ребятами, ехали из Калининграда, путь не ближний, 50 км)\nВ итоге приехали и ребята на баре развернули нас со словами «все столики заняты и у нас закончился табак» - настроение на вечер было нам обеспечено, уехли из Светлогорска с ужасным настроением. Спасибо, отличный сервис!\n\n"


Видим, что в таких случаях отзывы, как правило, отрицательные, есть сарказмы.
Можем использовать такие отзывы для разметки сарказмов.

In [28]:
data[(data['rating']==5) & (data['sentiment']=='negative')]['text'].head()

,text
161,"Уютно, чисто, комфортно. \nПожелание на завтрак поставить зерновую кофемашину, пить растворимый как то уже не привычно.\n"
182,"Нам очень понравитось данное заведение, ходили парой попробовать напиток чай с тапиокой, но не обошлось без покупки вкусняшек) взяли корн дог и ролл из рисовой бумаги и креветок. Безумно вкусно, что напиток, что еда. Очень уютная и приятная атмосфера. Месторасположение тоже огонь, прям в центре на Вайнера. Цены приемлемые. Хозяева совсем приятные, доброжелательные люди. \nПосле посещения в голове остались мысли вернуться туда снова и попробовать, что то новенькое. \n"
280,"Приятное кафе, где можно вкусно позавтракать, выпить кофе и не только. \nОчень проработанный интерьер, который только добавляет плюсов заведению. \nТакже есть выбор красивых (про вкусных не знаю, не пробовала) десертов\nЕсли думаете завтракать тут или нет, то однозначно советую!\n"
296,"Очень вкусно и сытно, рекомендую, быстрая подача\n"
342,"Уютное место в центре города. Еда вкусная. Быстро все принесли. Ресторан маленький, видимо поэтому официанты при входе спрашивают всех был ли забронирован стол. Если нет, то сажают, но ограничивают по времени\n"


В примерах с негативным окрасом и высокой оценкой отзывы положительные. В целом, оценки соответствуют разметке по модели тональности. Исключения - 4% отзывов. Для повышения точности разметки по ним можно просмотреть тексты отзывов вручную.

Для разметки сарказмов планируется вручную разметить небольшую выборку с положительной тональностью и низкой оценкой и на ней обучить модель.

In [32]:
## Сделаем разметку эмоций
classifier = pipeline(
    task="text-classification",
    model="seara/rubert-tiny2-russian-emotion-detection-cedr",
    device=0 if torch.cuda.is_available() else -1
)

label_mapping = {
    'no_emotion': 'нейтрально',
    'joy': 'радость',
    'sadness': 'грусть',
    'anger': 'злость',
    'surprise': 'удивление',
    'fear': 'страх'
}

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/117M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: seara/rubert-tiny2-russian-emotion-detection-cedr
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [33]:
def process_emotions_in_batches(texts, batch_size=64):
    results_labels = []
    results_scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        predictions = classifier(batch, truncation=True, max_length=512)

        for pred in predictions:
            results_labels.append(label_mapping.get(pred['label'], pred['label']))
            results_scores.append(round(pred['score'], 3))

    return results_labels, results_scores

In [34]:
texts_list = data['text'].tolist()
labels, scores = process_emotions_in_batches(texts_list, batch_size=64)

  0%|          | 0/2402 [00:00<?, ?it/s]

In [35]:
data['emotion'] = labels
data['emotion_confidence'] = scores

In [36]:
data.head()

,address,name_ru,rating,rubrics,text,org_type,sentiment,sentiment_confidence,emotion,emotion_confidence
6,"Воронежская область, Богучарский район, М-4 Дон, 770-й километр",У тещи,4.0,Кафе,"Глубинка страны во всех своих проявлениях. Ассортимент столовский, интерьер, качество и цены приемлемые для средней бюджетной столовой СССР, не все чистое, все не новое... А что бы вы хотели на трассе. Поел, желудок не бастовал, значит риск был оправдан. Номер для ночлега аналогичного толка. Пластиковое окно нормально не закрылось, шторы на окне нет, только тюль. Для первого этажа - плохо. Мало ли кто заглядывает. Туалет, душ... Совок, но, повторюсь, для трассы да за 1000 руб - сойдет.\n",Ресторан,neutral,0.612,нейтрально,0.979
8,"Москва, 4-й Кожевнический переулок, 4",Jinju,5.0,Кафе;Кофейня,"5 из 5🖤 Пил кофе и в Риме, и в Париже, но вкуснее, чем капуч на фундучном молоке с фирменными сливками Джинжу, не пробовал ничего! Десерты тоже очень необычные. Ребята - бариста большие молодцы! \n\nЧто можно улучшить? Маловато места, с посадкой можно что-то придумать?\n",Ресторан,positive,0.503,нейтрально,0.579
9,"Москва, 4-й Кожевнический переулок, 4",Jinju,4.0,Кафе;Кофейня,"Не очень удобное расположение, от метро идти мин 20 быстрым шагом через промзону. В самом кофе мест очень мало, а желающих очень много(( пирожные очень вкусные, кофе…бывает вкуснее. Второй раз именно туда на пойду.\n",Ресторан,negative,0.725,грусть,0.936
13,"Краснодарский край, городской округ Сочи, посёлок городского типа Дагомыс, Балтийская улица, 19",Пандок,2.0,Ресторан,"Самый большой плюс это месторасположение, набережная , шикарный вид на море! Красиво, уютно, вот собственно плюсы закончились .. огорчает отношение к посетителям, официанты неприветливые, не здравствуйте вам, не до свидания . Лица недовольные, неприятные, больше не хочется смотреть на такие! Кухня тоже оставляет желать лучшего, в люле кебаб кости попадаются, шашлык из говядины сухой и невкусный. Мы на отдыхе , на позитиве , денег не жалеем, но хочется приходить туда где нам рады!!\n",Ресторан,negative,0.762,радость,0.857
14,"Краснодарский край, городской округ Сочи, посёлок городского типа Дагомыс, Балтийская улица, 19",Пандок,5.0,Ресторан,"Добрый день! Сегодня во второй раз посетили с дочкой ресторан, нравится меню (куриный суп-лапша, пицца 4 сыра, яблочный штрудель; сациви из курицы, облепиховый чай-то, что успели попробовать за эти 2 посещения) и особенно порадовало прекрасное обслуживание, спасибо Алле за заботу и предупредительность! Разумное соотношение цены и качества, приятная атмосфера в заведении вызывают желание снова посетить и порекомендовать это место. Спасибо, и успехов! С уважением, Оксана и Злата.\n",Ресторан,positive,0.983,радость,0.865


In [38]:
data.groupby(['rating', 'emotion'])['name_ru'].count().unstack(fill_value=0)

emotion,грусть,злость,нейтрально,радость,страх,удивление
rating,,,,,,
0.0,1,0,30,39,0,0
1.0,770,336,5585,1762,111,131
2.0,329,93,3072,1183,19,64
3.0,533,89,5599,2343,9,117
4.0,622,53,9495,6369,17,133
5.0,1175,93,33586,79137,30,525


In [42]:
data[(data['rating']==5) & (data['emotion'] == 'страх')].head()

,address,name_ru,rating,rubrics,text,org_type,sentiment,sentiment_confidence,emotion,emotion_confidence
6921,"Краснодар, улица Архитектора Ишунина, 6",Чих-Пых,5.0,Кафе;Кальян-бар,"Уютная атмосфера, не громкая музыка, интересный современный интерьер в стиле лофт. Продуманное освещение. Хорошая вентиляция. Были опасения, что будет душно в цокольном этаже, наличие притяжной вентиляции полностью решает эту проблему. Приветливый персонал, серьезная публика. Было приятно провести вечер. Кухню попробовать не удалось.\nНемного не привычно: счёт на столик не приносят, расплачиваться надо в баре.\n",Ресторан,positive,0.625,страх,0.641
13943,"Республика Мордовия, Саранск, Большевистская улица, 60",Big Pig,5.0,"Ресторан;Бар, паб;Кафе","Видя столько положительных отзывов, было страшновато идти в этот ресторан, а вдруг ожидания не оправдаются?🤔\nНо к счастью, нам очень все понравилось☺️\nБрали сет «фирменное свинство» - это нарезка разных видов свиных деликатесов👌 и «ушки гриль»- порадовали брутальной подачей, тоже оказалось очень вкусно, хотя я к ушкам была настроена скептически (никогда ранее не пробовала)\nЦены не самые низкие конечно, но оно того стоит🙌\n",Ресторан,negative,0.835,страх,0.514
14917,"Иркутская область, Бодайбо, улица Петра Поручикова, 4",Багира,5.0,Кафе,"Еда вкусная, но расположено очень не удобно, крутой спуск, а подняться на машине просто трэш. Припарковаться особо не где, рядом детская площадка не огорожена, опасно\n",Ресторан,positive,0.666,страх,0.442
15195,"Кабардино-Балкарская Республика, Эльбрусский район, А-158, участок Баксан - Эльбрус, 91-й километр",Хычинhouse,5.0,Кафе,"Поужинали здесь с друзьями) \nНе пугайтесь обстановки, здесь в одном месте сельский магазин и небольшое кафе. У нас фоном по телевизору шла «Кавказская пленница», по коленкам бегал котик, а запивали всё это мы домашним вином (Изабелла))\n\nБрали лагман и манты.\nПорции огромные! \nБыло очень вкусно и по домашнему) \n\n\n",Ресторан,neutral,0.764,страх,0.469
39954,"Санкт-Петербург, набережная реки Мойки, 26",Союз,5.0,Ресторан,"Супер место - очень атмосферно !\nОтличные официанты \nПотрясающего вкуса и вида блюда\nА цены - шок , очень низкие для центра \nТеперь это мой фавоюворит 👍\n",Ресторан,positive,0.775,страх,0.333


In [43]:
data[(data['rating']==5) & (data['emotion'] == 'злость')].head()

,address,name_ru,rating,rubrics,text,org_type,sentiment,sentiment_confidence,emotion,emotion_confidence
1380,"Краснодарский край, Анапа, улица Ленина, 129",Космос,5.0,Кондитерская;Пекарня;Кафе,"Самые лучшие улитки с белковым кремом !!!!!!!!!!!!!!!!! Завоз каждые 4 часа , КОСМОС,когда откроетесь в НОВОРОССИЙСКЕ?!?!?!?!???!????????😍😍😍😍😍😍😍😍😍😍😍😍\n",Ресторан,neutral,0.538,злость,0.343
12147,"Санкт-Петербург, улица Льва Толстого, 1-3",Ketch Up,5.0,Ресторан,Стильный ресторан. Самые крутые бургеры здесь. После бургеров в кетчап. На бургеры других ресторанов смотришь с презрением. \n,Ресторан,positive,0.704,злость,0.764
14426,"Калужская область, Обнинск, проспект Маркса, 50А",Самсахона № 1,5.0,Кафе;Ресторан,"Бывал там пару раз,встречают как будто к родине в гости зашло,еда достойная такому заведению,цены сами понимаете кушать все хотят и повора и афицтаны!\n",Ресторан,negative,0.934,злость,0.429
16494,"Калининград, Красносельская улица, 60",Островок,5.0,Гостиница,Дёшево и сердито\n\n,Гостиница,neutral,0.757,злость,0.499
17550,"Ставропольский край, Кисловодск, Курортный бульвар, 6Б",Веранда,5.0,Кафе;Ресторан;Банкетный зал,"Бываем здесь каждый раз, когда приезжаем в Кисловодск. В основном из-за певицы и саксофонистки Ирины! Прелесть! Но и кухня,всегда на высоте, и обслуживание.\nДнем летом открываются окна-двери (раздвижные)-и получается ВЕРАНДА. В холодное время года кафе с панорамным стёклами, входящими га Курортный бульвар.\n\n",Ресторан,positive,0.634,злость,0.266


In [44]:
data[(data['rating']==1) & (data['emotion'] == 'страх')].head()

,address,name_ru,rating,rubrics,text,org_type,sentiment,sentiment_confidence,emotion,emotion_confidence
6104,"Саратов, улица имени А.Н. Радищева, 24",Восточный экспресс,1.0,Быстрое питание;Кафе;Доставка еды и обедов,"Осторожно! В заведении ротавирус. Руководство и весь их персонал переболели и сами сказали нам об этом, при этом не признали что мы заразились от них. Мы поели там и слегли на 4 дня! Не рекомендую есть там никому! \n",Ресторан,neutral,0.641,страх,0.327
8745,"Московская область, Одинцовский городской округ, Голицыно, Коммунистический проспект, 3, стр. 1",Чайхана Зам-зам,1.0,Кафе;Быстрое питание;Ресторан,Я в шоке от такой персонала. В меню показына что есть шурпа пол порции когда подошла спрашиваю говорит нет пол порции шурпа. Целый Казан шурпа перед моей глаза стоить. Повар или кто не понятно меня чуть не убил. Всегда приходили Всегда одавали по меню в это раз выгонял. Я конечно не ожидала такого. Может я в положение и у меня мало деньги хотела купить то что захотелось и на что деньги хватило \n,Ресторан,negative,0.909,страх,0.441
16793,"Московская область, Пушкино, Красноармейское шоссе, 91",Вкусно — и точка,1.0,Быстрое питание,"Это ужас какой то… 2-ой раз заезжаем в этот мак. Время ожидания заказа в первый раз 20 мин. Во второй раз 43!!!!! 43 минуты, Карл!!!! В туалете как в ощественном((((\n",Ресторан,negative,0.996,страх,0.735
24398,"Москва, Шарикоподшипниковская улица, 13, стр. 1",KFC,1.0,Быстрое питание,"Ужасно. Полно подозрительных личностей, душно, персонал ели живой, сделал заказ потом смотрю на монитор и вижу перед собой еще 10-12 заказов, подождал 5-7 минут и понимаю, что ждать еще дооооолго, а потом в спешке быстрее есть так как время поджимает. Тот случай когда зашел в фаст фуд и убежал от ожидания. Лучше не ходить\n",Ресторан,negative,0.852,страх,0.759
26725,"Московская область, Дмитровский городской округ, Яхрома, микрорайон Левобережье, 49, стр. 1",Вкусно — и точка,1.0,Быстрое питание,"Это просто ужас! Заехал вчера на машине, изначально планировал в макавто, подъезжаю, стоит по одной машине на каждой стойке, на 1 стойке машина стоит 5 минут, на 2 секунд 30, на 1 стойке водитель говорит, что его попросили подождать 2 минуты и замолчали, на 2 тихо. Подождали 5 минут, нифига не произошло, поставили машины, пошли внутрь, сделали заказ на терминале и ждём, спустя минус 7 наконец-то загарается номер моего заказа, но почему заказа на стойке нет, я подхожу, спрашиваю, в ответ тишина, ждал ещё минуты 4, пока мне его выдадут. Отвратительно!!! Сотрудники в это время спокойно болтают, ржут над водителями которые собираются и которых не обслуживают. Классно, не так ли? \nБургер собран на отвали. В коробке от биг спешл была какая-то черная фигня. \nОщущение от похода ужасные.\n",Ресторан,negative,0.925,страх,0.543


Разметка эмоций по базовой модели не всегда дает корректный результат. Для высоких оценок злость и страх часто не соответствуют отзыву. Для уточнения разметки можно попробовать другие модели + ручную разметку.

### Формирование выборки и стратегия валидации

Для обучения выберем 80% выборки в каждой категории (отели, растораны) с кросс-валидацией на 5 фолдов и стратификацией по оценкам. Оставшиеся 20% пойдут на тестовую выборку.